In [1]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/SnowPole_Detection_Dataset/
# !git clone https://github.com/ultralytics/ultralytics.git

/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset


In [3]:
!pip uninstall -y ultralytics

In [4]:
!rm -rf /content/ultralytics
!git clone https://github.com/MuhammadIbneRafiq/ultralytics4channel /content/ultralytics

Cloning into '/content/ultralytics'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (276/276), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 276 (delta 65), reused 254 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (276/276), 706.61 KiB | 1.57 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [5]:
!pip install -q ultralytics==8.2.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 53.9 MB/s eta 0:00:00


In [6]:
import sys
sys.path.insert(0, "/content")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: /content/ultralytics/__init__.py


In [7]:
import torch
from ultralytics import YOLO
from pathlib import Path
import shutil
import cv2
import numpy as np
import yaml
from tqdm import tqdm

from ultralytics import YOLO
import torch

from torch.utils.data import Dataset, DataLoader

In [8]:
COMB_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/images")

# 1-channel range-normalized images
RANGE_ROOT = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous")

# New 4-channel dual-input dataset
DUAL_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range")
DUAL_ROOT.mkdir(parents=True, exist_ok=True)

print("COMB_ROOT :", COMB_ROOT)
print("RANGE_ROOT:", RANGE_ROOT)
print("DUAL_ROOT :", DUAL_ROOT)

COMB_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/images
RANGE_ROOT: /content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous
DUAL_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range


In [9]:
# !yolo train model=yolov9t.pt epochs=150 imgsz=1024 device=0 batch=2 data=/content/drive/MyDrive/data.yaml project=/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec

In [10]:
# !yolo val \
#   model=/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec-v11n/train2/weights/best.pt \
#   data=/content/drive/MyDrive/data.yaml \
#   split=test \
#   imgsz=1024 \
#   device=0 \
#   batch=16 \
#   project="comb5_signal_reflec_range_11n" \
#   name="comb5_signal_reflec_range_11n_test_eval"


In [11]:
def make_dual_split_npy(split: str,
                        comb_root: Path,
                        range_root: Path,
                        dual_root: Path,
                        save_as_png: bool = True):
    """
    Create 4-channel dual images by stacking comb RGB (BGR) + range (.npy float32 [0,1]).
    - comb_root: root containing comb_root/images/<split>/*.png and comb_root/labels/<split>/*.txt
    - range_root: root containing range_root/<split>/*.npy (each named like the comb image stem)
    - dual_root: destination root; will create dual_root/images/<split> and dual_root/labels/<split>
    - save_as_png: if True, save stacked RGBA PNGs (4 channel) so existing YOLO loaders can read them.
                   (range channel is quantized to uint8 for the PNG; the original .npy is left unchanged)
    """
    comb_img_dir   = comb_root / split
    src_lbl_dir    = comb_root / "../" / "labels" / split
    range_npy_dir  = range_root / split
    dual_img_dir   = dual_root / "images" / split
    dual_lbl_dir   = dual_root / "labels" / split

    # if this split already has images, skip doing anything
    if dual_img_dir.exists() and any(dual_img_dir.glob("*.png")):
        print(f"[{split}] dual images already exist in {dual_img_dir}, skipping.")
        return

    dual_img_dir.mkdir(parents=True, exist_ok=True)
    dual_lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels from comb labels to dual labels (they are the same)
    if src_lbl_dir.exists():
        for lbl in src_lbl_dir.glob("*.txt"):
            # copy2 preserves timestamps, etc.
            shutil.copy2(lbl, dual_lbl_dir / lbl.name)
    else:
        print(f"Warning: source label dir not found: {src_lbl_dir}")

    # gather comb images
    img_files = sorted(comb_img_dir.glob("*.*"))
    print(f"[{split}] comb images found: {len(img_files)}")

    for comb_path in tqdm(img_files, desc=f"make_dual_split ({split})"):
        stem = comb_path.stem

        # read comb RGB (OpenCV: BGR)
        comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
        if comb is None:
            print("Could not read comb image:", comb_path)
            continue

        # read corresponding range .npy
        range_npy_path = range_npy_dir / f"{stem}.npy"
        if not range_npy_path.exists():
            # try alternative stem patterns if needed
            print(f"Missing range .npy for {stem} -> {range_npy_path} (skipping)")
            continue

        try:
            range_arr = np.load(str(range_npy_path))   # expected float32 in [0,1]
        except Exception as e:
            print(f"Failed to load {range_npy_path}: {e}")
            continue

        # squeeze any extra dims
        if range_arr.ndim == 3 and range_arr.shape[0] in (1,):
            range_arr = np.squeeze(range_arr, axis=0)
        if range_arr.ndim != 2:
            # if it has a channel dim (H,W,1) -> squeeze
            if range_arr.ndim == 3 and range_arr.shape[2] == 1:
                range_arr = np.squeeze(range_arr, axis=2)
            else:
                print(f"Unexpected shape for range npy {range_npy_path}: {range_arr.shape} (skipping)")
                continue

        # ensure float32 and clip to [0,1]
        range_arr = range_arr.astype(np.float32)
        range_arr = np.clip(range_arr, 0.0, 1.0)

        # convert range to uint8 for stacking if saving PNGs (visualization/training with standard loader)
        range_uint8 = (range_arr * 255.0).astype(np.uint8)

        # resize range to comb dims if necessary (note cv2 resize expects (width, height))
        if range_uint8.shape != comb.shape[:2]:
            range_uint8 = cv2.resize(range_uint8, (comb.shape[1], comb.shape[0]), interpolation=cv2.INTER_NEAREST)

        # stack into 4-channel: B, G, R, RANGE
        rgba = np.dstack([comb, range_uint8])  # result dtype uint8, shape (H, W, 4)

        # write stacked 4-channel PNG so YOLO-like image loaders can ingest it
        if save_as_png:
            out_img_path = dual_img_dir / f"{stem}.png"
            # OpenCV will write all 4 channels to PNG when given a 4-channel array.
            cv2.imwrite(str(out_img_path), rgba)

    print(f"[{split}] done. Dual images written to {dual_img_dir}, labels copied to {dual_lbl_dir}")


# Run for all splits (call this cell)
for split in ["train", "valid", "test"]:
    make_dual_split_npy(split, comb_root=COMB_ROOT, range_root=RANGE_ROOT, dual_root=DUAL_ROOT, save_as_png=True)


[train] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/train, skipping.
[valid] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/valid, skipping.
[test] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/test, skipping.


In [12]:
ORIG_DATA_YAML = COMB_ROOT / "../" /"data.yaml"
DUAL_DATA_YAML = DUAL_ROOT / "data.yaml"

# with open(ORIG_DATA_YAML, "r") as f:
#     cfg = yaml.safe_load(f)

# base = DUAL_ROOT

# def make_rel(p):
#     # p might be absolute or relative – we point to new dual root
#     p = Path(p)
#     return str((base / "images" / p.name).parent)  # keep split names

# # If your original yaml used explicit paths, you can instead do:
# # cfg["path"]  = str(DUAL_ROOT)
# cfg["path"]  = str(DUAL_ROOT)
# cfg["train"] = "images/train"
# cfg["valid"]   = "images/valid"
# cfg["test"]  = "images/test"
# cfg["channels"] = 4          # tell YOLO this is 4-channel data with the RGB-Alpha

# with open(DUAL_DATA_YAML, "w") as f:
#     yaml.safe_dump(cfg, f)

# print(DUAL_DATA_YAML.read_text())

In [ ]:
from ultralytics import YOLO
import torch
from ultralytics.nn.tasks import DetectionModel

import torch
from ultralytics.nn.tasks import DetectionModel
# from ultralytics.nn.modules.block import ELAN1, AConv, RepNCSPELAN4, SPPELAN

# allow the DetectionModel class for unpickling

torch.serialization.add_safe_globals([
    DetectionModel,
    # ELAN1,
    # AConv,
    # RepNCSPELAN4,
    # SPPELAN,
])


model = YOLO("yolov8n.pt")  # or YOLO("yolov9t.pt", task="detect")
model.model.eval()

model.train(
    data=str(DUAL_DATA_YAML),
    epochs=400,
    imgsz=1024,
    device='cpu',
    batch=16,
    project="dual_comb_range_experiments",
    name="dual_comb_rgb_plus_range_v9t_4ch_825",
)


New https://pypi.org/project/ultralytics/8.3.241 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.5 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/data.yaml, epochs=400, time=None, patience=100, batch=16, imgsz=1024, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=dual_comb_range_experiments, name=dual_comb_rgb_plus_range_v9t_4ch_8258, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=Fa

/content/ultralytics/engine/trainer.py:262: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
train: Scanning /content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/train... 816 images, 0 backgrounds, 0 corrupt:  60%|█████▉    | 816/1367 [00:38<01:06,  8.27it/s]